# Seam Residual Corrector v1 (Colab)

**Контракт:** этот ноутбук **не зависит** от копии репозитория на вашем компьютере. Исходники для train/eval/export подтягиваются **только** так:

1. **`GITHUB_OWNER_REPO`** — кратко `user/repo` (подставит `REPO_CLONE_URL` автоматически);
2. **`REPO_CLONE_URL`** — публичный (или с токеном) `git clone` в `PROJECT_ROOT`;
3. **`RUNTIME_ZIP_URL`** — прямой URL на zip-архив исходников (например `https://github.com/USER/REPO/archive/refs/heads/main.zip`).

Ровно один из вариантов обязателен в ячейке **PARAMS** (часто достаточно `GITHUB_OWNER_REPO`).

**Данные** (материализованный strip-cache, манифесты, конфиги в bundle) — с **Google Drive**: заархивируйте локально `package_training_bundle` и загрузите `.tar.gz`, путь укажите в `DATASET_BUNDLE_DRIVE_PATH`. Без датасета обучение не запустится.

Пайплайн: распаковка bundle → **fetch кода** → валидация layout → runtime yaml → train → **10b TensorBoard (ссылка с порта)** → eval → export → verify → синк на Drive.

**TensorBoard:** логи пишет `train_resunet` в `outputs/logs/tensorboard` (= `LOCAL_TENSORBOARD`). Ячейка **10b** поднимает TensorBoard в фоне на **порту 6006**, проверяет `http://127.0.0.1:6006`, затем вызывает `google.colab.output.serve_kernel_port_as_window` — **кликабельная ссылка в новой вкладке** (встроенный iframe в Colab часто серый). Можно повторно запустить 10b во время/после train. Копия events на Drive: `RUN_NAME/tensorboard/`. Установка `protobuf<5` в 10b — намеренно (совместимость с TensorBoard в Colab).


In [ ]:
# 0. PARAMS
from pathlib import Path
import os

DATASET_BUNDLE_DRIVE_PATH = '/content/drive/MyDrive/unet_seam/seam_residual_corrector_training_bundle.tar.gz'
DRIVE_RUNS_DIR = '/content/drive/MyDrive/unet_seam_runs'
RUN_NAME = 'seam_residual_corrector_v1_run001'
USE_RAMDISK = True
RAMDISK_SIZE_GB = 48
COPY_ARCHIVE_TO_RAM_FIRST = True
SYNC_INTERVAL_SEC = 180
TRAIN_BATCH_SIZE = 32
TRAIN_EPOCHS = 20
TRAIN_NUM_WORKERS = 16
VAL_BATCH_SIZE = 8
PRIMARY_CHECKPOINT = 'best_boundary_ciede2000.pt'
PROJECT_ROOT = Path('/content/seam_runtime')
LOCAL_OUTPUTS = PROJECT_ROOT / 'outputs'
LOCAL_CHECKPOINTS = LOCAL_OUTPUTS / 'checkpoints'
LOCAL_EVAL = LOCAL_OUTPUTS / 'eval_reports'
LOCAL_EXPORTS = LOCAL_OUTPUTS / 'exports'
LOCAL_LOGS = LOCAL_OUTPUTS / 'logs'
LOCAL_DATA_ROOT = Path('/content/dataset_bundle')
DRIVE_RUN_DIR = Path(DRIVE_RUNS_DIR) / RUN_NAME
DRIVE_CKPT_DIR = DRIVE_RUN_DIR / 'checkpoints'
DRIVE_EVAL_DIR = DRIVE_RUN_DIR / 'eval_reports'
DRIVE_EXPORT_DIR = DRIVE_RUN_DIR / 'exports'
DRIVE_LOG_DIR = DRIVE_RUN_DIR / 'logs'
# Исходники: задай GITHUB_OWNER_REPO (одна строка) ИЛИ REPO_CLONE_URL ИЛИ RUNTIME_ZIP_URL.
# Локальный репозиторий на диске не нужен.
GITHUB_OWNER_REPO = "aaandreyev/unet_seam"  # public repo: подставит REPO_CLONE_URL; None — если только zip/свой URL
REPO_CLONE_URL = None  # например 'https://github.com/USER/unet_seam.git' (для private — с токеном в URL)
REPO_CLONE_REF = "main"
RUNTIME_ZIP_URL = None  # например 'https://github.com/USER/unet_seam/archive/refs/heads/main.zip'
if GITHUB_OWNER_REPO and not (REPO_CLONE_URL or RUNTIME_ZIP_URL):
    REPO_CLONE_URL = f"https://github.com/{GITHUB_OWNER_REPO}.git"


In [ ]:
# 1. MOUNT DRIVE
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted')


In [ ]:
# 2. INSTALL / RUNTIME CHECKS
import os, sys, subprocess, importlib.util, platform, json
pkgs = ['pyyaml','scipy','scikit-image','safetensors','tqdm','lpips','psutil']
subprocess.run(['apt-get', 'update', '-qq'], check=False)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'pigz'], check=False)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'git'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)
import torch
device = 'cuda' if torch.cuda.is_available() else 'mps' if getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available() else 'cpu'
print(json.dumps({'python': sys.executable, 'platform': platform.platform(), 'torch': torch.__version__, 'device': device}, ensure_ascii=False))


In [ ]:
# 3. OPTIONAL RAMDISK
import os, psutil, subprocess, shutil
RAM_ROOT = Path('/content/ramdisk')
if USE_RAMDISK:
    mem = psutil.virtual_memory()
    eff = min(RAMDISK_SIZE_GB, max(8, int((mem.available / (1024**3)) * 0.5)))
    RAM_ROOT.mkdir(parents=True, exist_ok=True)
    mounted = subprocess.run(['mountpoint', str(RAM_ROOT)], capture_output=True).returncode == 0
    if not mounted:
        subprocess.run(['mount', '-t', 'tmpfs', '-o', f'size={eff}G,mode=777', 'tmpfs', str(RAM_ROOT)], check=True)
    DATA_ROOT = RAM_ROOT / 'dataset_bundle'
else:
    DATA_ROOT = LOCAL_DATA_ROOT
DATA_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_OUTPUTS.mkdir(parents=True, exist_ok=True)
for p in [LOCAL_CHECKPOINTS, LOCAL_EVAL, LOCAL_EXPORTS, LOCAL_LOGS]:
    p.mkdir(parents=True, exist_ok=True)
print('DATA_ROOT =', DATA_ROOT)


In [ ]:
# 4. EXTRACT DATASET BUNDLE FROM DRIVE
import shutil, tarfile, time
from tqdm.auto import tqdm

src = Path(DATASET_BUNDLE_DRIVE_PATH)
if not src.exists():
    raise FileNotFoundError(src)
archive_local = src
if COPY_ARCHIVE_TO_RAM_FIRST and USE_RAMDISK:
    archive_local = RAM_ROOT / src.name
    with src.open('rb') as fin, archive_local.open('wb') as fout:
        total = src.stat().st_size
        with tqdm(total=total, unit='B', unit_scale=True, desc='copy_archive') as pbar:
            while True:
                chunk = fin.read(8 * 1024 * 1024)
                if not chunk:
                    break
                fout.write(chunk)
                pbar.update(len(chunk))
if DATA_ROOT.exists():
    shutil.rmtree(DATA_ROOT)
DATA_ROOT.mkdir(parents=True, exist_ok=True)
with tarfile.open(archive_local, 'r:*') as tf:
    members = tf.getmembers()
    for m in tqdm(members, desc='extract_bundle', dynamic_ncols=True):
        tf.extract(m, DATA_ROOT)
print('bundle extracted to', DATA_ROOT)


In [ ]:
# 5. FETCH PROJECT FROM GIT OR ZIP (self-contained policy: no embedded repo copy in the notebook)
import os
import shutil
import subprocess
import sys
import urllib.request
import zipfile
from pathlib import Path


def _clone_repo() -> bool:
    if not REPO_CLONE_URL:
        return False
    if PROJECT_ROOT.exists():
        shutil.rmtree(PROJECT_ROOT)
    PROJECT_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "-b",
            REPO_CLONE_REF,
            REPO_CLONE_URL,
            str(PROJECT_ROOT),
        ],
        check=True,
    )
    return True


def _unpack_zip() -> None:
    if not RUNTIME_ZIP_URL:
        raise ValueError("RUNTIME_ZIP_URL is empty")
    if PROJECT_ROOT.exists():
        shutil.rmtree(PROJECT_ROOT)
    zpath = Path("/tmp/unet_seam_runtime_src.zip")
    print("downloading:", RUNTIME_ZIP_URL)
    urllib.request.urlretrieve(RUNTIME_ZIP_URL, zpath)
    tmp = Path("/tmp/unet_seam_src_extract")
    if tmp.exists():
        shutil.rmtree(tmp)
    tmp.mkdir(parents=True)
    with zipfile.ZipFile(zpath, "r") as zf:
        zf.extractall(tmp)
    top = [p for p in tmp.iterdir() if p.is_dir()]
    if len(top) != 1:
        raise RuntimeError("zip must contain exactly one top-level directory, got: " + repr([p.name for p in top]))
    shutil.move(str(top[0]), str(PROJECT_ROOT))


if _clone_repo():
    pass
elif RUNTIME_ZIP_URL:
    _unpack_zip()
else:
    raise ValueError(
        "В ячейке PARAMS задай исходники: GITHUB_OWNER_REPO = 'user/unet_seam' "
        "или REPO_CLONE_URL, или RUNTIME_ZIP_URL. Репо в ноутбуке не вшито — код только с git/zip."
    )

sys.path.insert(0, str(PROJECT_ROOT))
_n_py = sum(1 for _ in PROJECT_ROOT.rglob("*.py"))
print("PROJECT_ROOT =", PROJECT_ROOT, "| python files =", _n_py)


In [ ]:
# 6. VALIDATE BUNDLE LAYOUT
import json
required = [
    DATA_ROOT / 'manifests/strip_train_cache.jsonl',
    DATA_ROOT / 'manifests/strip_val_cache.jsonl',
    DATA_ROOT / 'outputs/strip_cache/train',
    DATA_ROOT / 'outputs/strip_cache/val',
]
for p in required:
    if not p.exists():
        raise FileNotFoundError(p)
print(json.dumps({
    'train_cache_dirs': len([p for p in (DATA_ROOT / 'outputs/strip_cache/train').iterdir() if p.is_dir()]),
    'val_cache_dirs': len([p for p in (DATA_ROOT / 'outputs/strip_cache/val').iterdir() if p.is_dir()]),
}, ensure_ascii=False))


In [ ]:
# 7. BUILD RUNTIME CONFIGS
import yaml, json, os, shutil
cfg_dir = PROJECT_ROOT / 'runtime_configs'
cfg_dir.mkdir(parents=True, exist_ok=True)
train_cfg = yaml.safe_load((PROJECT_ROOT / 'configs/train_synth_v1.yaml').read_text(encoding='utf-8'))
eval_cfg = yaml.safe_load((PROJECT_ROOT / 'configs/eval_v1.yaml').read_text(encoding='utf-8'))
export_cfg = yaml.safe_load((PROJECT_ROOT / 'configs/export_v1.yaml').read_text(encoding='utf-8'))
train_cfg['dataset']['cache_root'] = str(DATA_ROOT / 'outputs/strip_cache')
train_cfg['dataset']['train_cache_manifest'] = str(DATA_ROOT / 'manifests/strip_train_cache.jsonl')
train_cfg['dataset']['val_cache_manifest'] = str(DATA_ROOT / 'manifests/strip_val_cache.jsonl')
train_cfg['train']['batch_size'] = TRAIN_BATCH_SIZE
train_cfg['train']['num_epochs'] = TRAIN_EPOCHS
train_cfg['train']['num_workers'] = TRAIN_NUM_WORKERS
eval_cfg['checkpoint'] = str(LOCAL_CHECKPOINTS / PRIMARY_CHECKPOINT)
eval_cfg['report_root'] = str(LOCAL_EVAL)
eval_cfg['cache_root'] = str(DATA_ROOT / 'outputs/strip_cache')
eval_cfg['val_cache_manifest'] = str(DATA_ROOT / 'manifests/strip_val_cache.jsonl')
export_cfg['checkpoint'] = str(LOCAL_CHECKPOINTS / PRIMARY_CHECKPOINT)
export_cfg['export_root'] = str(LOCAL_EXPORTS)
(cfg_dir / 'train.yaml').write_text(yaml.safe_dump(train_cfg, sort_keys=False), encoding='utf-8')
(cfg_dir / 'eval.yaml').write_text(yaml.safe_dump(eval_cfg, sort_keys=False), encoding='utf-8')
(cfg_dir / 'export.yaml').write_text(yaml.safe_dump(export_cfg, sort_keys=False), encoding='utf-8')
print('runtime configs written to', cfg_dir)


In [ ]:
# 8. OPTIONAL RESUME FROM DRIVE LAST CHECKPOINT
import shutil
resume_path = None
DRIVE_CKPT_DIR.mkdir(parents=True, exist_ok=True)
drive_last = DRIVE_CKPT_DIR / 'last.pt'
if drive_last.exists():
    LOCAL_CHECKPOINTS.mkdir(parents=True, exist_ok=True)
    local_resume = LOCAL_CHECKPOINTS / 'resume_last.pt'
    shutil.copy2(drive_last, local_resume)
    resume_path = local_resume
print('resume_path =', resume_path)


In [ ]:
# 9. BACKGROUND SYNC TO DRIVE
import threading, time, shutil
SYNC_STOP = threading.Event()
for p in [DRIVE_RUN_DIR, DRIVE_CKPT_DIR, DRIVE_EVAL_DIR, DRIVE_EXPORT_DIR, DRIVE_LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)
def sync_tree(src: Path, dst: Path):
    if not src.exists():
        return
    for path in src.rglob('*'):
        if not path.is_file():
            continue
        rel = path.relative_to(src)
        out = dst / rel
        out.parent.mkdir(parents=True, exist_ok=True)
        if not out.exists() or out.stat().st_mtime < path.stat().st_mtime or out.stat().st_size != path.stat().st_size:
            shutil.copy2(path, out)
def sync_loop():
    while not SYNC_STOP.wait(SYNC_INTERVAL_SEC):
        sync_tree(LOCAL_CHECKPOINTS, DRIVE_CKPT_DIR)
        sync_tree(LOCAL_EVAL, DRIVE_EVAL_DIR)
        sync_tree(LOCAL_EXPORTS, DRIVE_EXPORT_DIR)
        sync_tree(LOCAL_LOGS, DRIVE_LOG_DIR)
sync_thread = threading.Thread(target=sync_loop, daemon=True)
sync_thread.start()
print('background sync started')


In [ ]:
# 10. TRAIN
import os, sys, subprocess, json
env = os.environ.copy()
env['PYTHONPATH'] = str(PROJECT_ROOT)
env['PYTHONUNBUFFERED'] = '1'
cmd = [sys.executable, '-u', '-m', 'scripts.train_resunet', '--config', str(PROJECT_ROOT / 'runtime_configs/train.yaml')]
if resume_path is not None:
    cmd += ['--resume', str(resume_path)]
print('TRAIN CMD:', ' '.join(map(str, cmd)))
print('Лог: JSON с event=train_start / train_step / epoch_end; кривые — в ячейке 10b (TensorBoard).')
subprocess.run(cmd, cwd=str(PROJECT_ROOT), env=env, check=True)


In [ ]:
# 11. EVAL
import os, sys, subprocess
env = os.environ.copy()
env['PYTHONPATH'] = str(PROJECT_ROOT)
env['PYTHONUNBUFFERED'] = '1'
cmd = [sys.executable, '-u', '-m', 'scripts.run_eval', '--config', str(PROJECT_ROOT / 'runtime_configs/eval.yaml')]
print('EVAL CMD:', ' '.join(map(str, cmd)))
subprocess.run(cmd, cwd=str(PROJECT_ROOT), env=env, check=True)


In [ ]:
# 12. EXPORT + VERIFY
import os, sys, subprocess
env = os.environ.copy()
env['PYTHONPATH'] = str(PROJECT_ROOT)
env['PYTHONUNBUFFERED'] = '1'
export_cmd = [sys.executable, '-u', '-m', 'scripts.export_safetensors', '--config', str(PROJECT_ROOT / 'runtime_configs/export.yaml')]
verify_cmd = [sys.executable, '-u', '-m', 'scripts.verify_export', '--config', str(PROJECT_ROOT / 'runtime_configs/export.yaml')]
print('EXPORT CMD:', ' '.join(map(str, export_cmd)))
subprocess.run(export_cmd, cwd=str(PROJECT_ROOT), env=env, check=True)
print('VERIFY CMD:', ' '.join(map(str, verify_cmd)))
subprocess.run(verify_cmd, cwd=str(PROJECT_ROOT), env=env, check=True)


In [ ]:
# 13. FINAL SYNC + SUMMARY
SYNC_STOP.set()
sync_tree(LOCAL_CHECKPOINTS, DRIVE_CKPT_DIR)
sync_tree(LOCAL_EVAL, DRIVE_EVAL_DIR)
sync_tree(LOCAL_EXPORTS, DRIVE_EXPORT_DIR)
sync_tree(LOCAL_LOGS, DRIVE_LOG_DIR)
print('Drive checkpoint files:', sorted(p.name for p in DRIVE_CKPT_DIR.glob('*')))
print('Drive export files:', sorted(p.name for p in DRIVE_EXPORT_DIR.glob('*')))
print('Drive eval runs:', sorted(p.name for p in DRIVE_EVAL_DIR.glob('*')))
